<a href="https://colab.research.google.com/github/RiyaThakur14/DATA_SCIENCE/blob/main/NLP_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NLP IMPLEMENTATION ON REVIEWS DATASET
# Dataset columns:
# review_id     -> Unique ID of each review
# review_text   -> Text written by the student/customer
# sentiment     -> Target column: Positive or Negative
# ============================================================


# ------------------------------------------------------------
# STEP 1: IMPORT FILE IN GOOGLE COLAB
# ------------------------------------------------------------

# This line imports the "files" tool from Google Colab.
# It helps us upload a file from our computer into Google Colab.
from google.colab import files

# This line opens a file upload button in Colab.
# You need to choose your CSV file, for example: reviews(1).csv
uploaded = files.upload()


# ------------------------------------------------------------
# STEP 2: IMPORT IMPORTANT PYTHON LIBRARIES
# ------------------------------------------------------------

# pandas is used to read and work with tabular data like CSV files.
import pandas as pd

# re means regular expression.
# It helps us clean text by removing unwanted symbols, numbers, etc.
import re

# train_test_split is used to divide the dataset into training and testing data.
# Training data teaches the model.
# Testing data checks how well the model learned.
from sklearn.model_selection import train_test_split

# TfidfVectorizer converts text into numbers.
# Machine learning models cannot understand words directly.
# So, TF-IDF gives importance scores to useful words.
from sklearn.feature_extraction.text import TfidfVectorizer

# LogisticRegression is a simple machine learning algorithm.
# Here, it will learn whether a review is Positive or Negative.
from sklearn.linear_model import LogisticRegression

# These metrics help us check the performance of our model.
# accuracy_score tells how many predictions are correct.
# classification_report gives more detailed performance.
from sklearn.metrics import accuracy_score, classification_report


# ------------------------------------------------------------
# STEP 3: READ THE CSV FILE
# ------------------------------------------------------------

# This line reads the uploaded CSV file.
# Make sure the file name is exactly same as the uploaded file name.
df = pd.read_csv("reviews.csv")

# This line shows the first 5 rows of the dataset.
# It helps us quickly check whether the file loaded correctly or not.
print(df.head())


# ------------------------------------------------------------
# STEP 4: CHECK BASIC INFORMATION ABOUT DATASET
# ------------------------------------------------------------

# This line shows the total number of rows and columns in the dataset.
# Example output: (8, 3) means 8 rows and 3 columns.
print("Shape of dataset:", df.shape)

# This line shows column names and data types.
# It helps us understand which columns are text, numbers, etc.
print(df.info())

# This line checks missing values in each column.
# Missing values can create problems during model training.
print(df.isnull().sum())


# ------------------------------------------------------------
# STEP 5: SELECT TEXT COLUMN AND TARGET COLUMN
# ------------------------------------------------------------

# review_text is the input column.
# This is the actual sentence/review written by the user.
X = df["review_text"]

# sentiment is the output column.
# This is what we want the model to predict: Positive or Negative.
y = df["sentiment"]


# ------------------------------------------------------------
# STEP 6: CLEAN THE TEXT DATA
# ------------------------------------------------------------

# This function will clean each review.
# Cleaning text is important in NLP because raw text may contain capital letters,
# symbols, punctuation, extra spaces, etc.
def clean_text(text):

    # Convert the text into lowercase.
    # Example: "Good Class" becomes "good class"
    # This helps Python treat "Good" and "good" as the same word.
    text = text.lower()

    # Remove anything that is not an English alphabet or space.
    # Example: "good!!!" becomes "good"
    # This removes punctuation, numbers, and special symbols.
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces from the beginning and end of the sentence.
    # Example: "  good class  " becomes "good class"
    text = text.strip()

    # Return the cleaned text.
    return text


# Apply the clean_text function on every review in review_text column.
# A new column called clean_review is created.
df["clean_review"] = df["review_text"].apply(clean_text)

# Show original review and cleaned review side by side.
# This helps us understand how text cleaning changed the data.
print(df[["review_text", "clean_review"]])


# ------------------------------------------------------------
# STEP 7: CONVERT TEXT INTO NUMBERS USING TF-IDF
# ------------------------------------------------------------

# Machine learning models cannot understand text directly.
# So, we use TfidfVectorizer to convert words into numerical values.
vectorizer = TfidfVectorizer()

# fit_transform does two things:
# 1. fit    -> learns all important words from the reviews
# 2. transform -> converts those words into numbers
X_vectorized = vectorizer.fit_transform(df["clean_review"])

# This line shows the shape of the converted text data.
# Rows represent reviews.
# Columns represent unique important words.
print("Shape after TF-IDF conversion:", X_vectorized.shape)


# ------------------------------------------------------------
# STEP 8: SPLIT DATA INTO TRAINING AND TESTING SETS
# ------------------------------------------------------------

# We split the data into train and test data.
# test_size=0.25 means 25% data will be used for testing.
# random_state=42 ensures same split every time we run the code.
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.25,
    random_state=42
)

# Show how many records are used for training and testing.
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


# ------------------------------------------------------------
# STEP 9: CREATE AND TRAIN THE MACHINE LEARNING MODEL
# ------------------------------------------------------------

# Create the Logistic Regression model.
# This model will learn the relationship between review words and sentiment.
model = LogisticRegression()

# Train the model using training data.
# X_train contains review words converted into numbers.
# y_train contains the correct sentiment labels.
model.fit(X_train, y_train)


# ------------------------------------------------------------
# STEP 10: MAKE PREDICTIONS ON TEST DATA
# ------------------------------------------------------------

# The model predicts sentiment for unseen test reviews.
y_pred = model.predict(X_test)

# Print the predicted sentiment values.
print("Predicted Sentiments:", y_pred)

# Print the actual sentiment values.
# This helps us compare prediction vs real answer.
print("Actual Sentiments:", list(y_test))


# ------------------------------------------------------------
# STEP 11: CHECK MODEL ACCURACY
# ------------------------------------------------------------

# accuracy_score compares actual values and predicted values.
# Example: accuracy 1.0 means 100% correct prediction.
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the model.
print("Model Accuracy:", accuracy)

# classification_report gives detailed result.
# precision, recall, and f1-score are useful metrics in classification.
print(classification_report(y_test, y_pred))


# ------------------------------------------------------------
# STEP 12: TEST THE MODEL WITH A NEW REVIEW
# ------------------------------------------------------------

# This is a new review written by a user.
# We want to check whether the model predicts it as Positive or Negative.
new_review = ["The teacher explained everything clearly and the project was very useful"]

# Clean the new review using the same cleaning function.
new_review_cleaned = [clean_text(review) for review in new_review]

# Convert the cleaned new review into numbers using the same TF-IDF vectorizer.
# We use transform, not fit_transform, because the vectorizer has already learned words earlier.
new_review_vectorized = vectorizer.transform(new_review_cleaned)

# Predict sentiment for the new review.
prediction = model.predict(new_review_vectorized)

# Print the final prediction.
print("New Review:", new_review[0])
print("Predicted Sentiment:", prediction[0])

Saving reviews.csv to reviews.csv
   review_id                                        review_text sentiment
0          1  I loved the online class! The teacher explaine...  Positive
1          2  The session was boring and the audio quality w...  Negative
2          3  Power BI dashboard training was very useful fo...  Positive
3          4  I did not understand the SQL joins topic prope...  Negative
4          5  The Excel project was interesting, practical, ...  Positive
Shape of dataset: (8, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    8 non-null      int64 
 1   review_text  8 non-null      object
 2   sentiment    8 non-null      object
dtypes: int64(1), object(2)
memory usage: 324.0+ bytes
None
review_id      0
review_text    0
sentiment      0
dtype: int64
                                         review_text  \
0  I loved t

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_